# Filter Metadata file 
Drop all Acne_NL samples that have a score bigger than 0

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

import biom
from biom import load_table
from qiime2 import Artifact
import qiime2 as q2

In [3]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Now filter out the samples that match 'low Acne_NL'
samples_to_drop = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']

# Get the number of samples that are being dropped
num_dropped_samples = len(samples_to_drop)

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Drop the samples
metadata_df = metadata_df[metadata_df['severity_group'] != 'low Acne_NL']

# Output the number of dropped samples
print(f'Number of samples dropped: {num_dropped_samples}')

Number of samples dropped: 23


In [4]:
# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

In [5]:
# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

In [ ]:
# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
        # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")

    # Transpose the filtered biom table
    biom_tbl_filtered_T = biom_tbl_filtered.transpose()

    # Convert the transposed BIOM table to a QIIME 2 Artifact
    biom_tbl_filtered_artifact = q2.Artifact.import_data('FeatureTable[Frequency]', biom_tbl_filtered_T)
    
    # Save the artifact as a .qza file
    biom_tbl_filtered_artifact.save(f'../Data/16S/CTF/filtered_{dataset_id}.qza')

In [10]:
# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/CTF/174951_rarefied_table_unannotated_absolute_abundances_T.qza'),
    ('174950', 'V1-V3', '../Data/16S/CTF/174950_rarefied_table_unannotated_absolute_abundances_T.qza')
]

In [12]:
# Loop over both tables
for dataset_id, region_title, qza_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the QIIME 2 artifact and extract the table data
    table_artifact = q2.Artifact.load(qza_file)
    table = table_artifact.view(biom.Table)
    
    # Filter the table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(table, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")

    # Convert the filtered table back to a QIIME 2 Artifact
    biom_tbl_filtered_artifact = q2.Artifact.import_data('FeatureTable[Frequency]', biom_tbl_filtered)
    
    # Save the artifact as a .qza file
    biom_tbl_filtered_artifact.save(f'../Data/16S/CTF/filtered_{dataset_id}_T_10252024.qza')


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD314.D28.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD319.D28.C3'}
Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD314.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD310.D

                   GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT  \
LAMI.RD001.D0.C1                                                11.0                                                                                                        
LAMI.RD001.D0.C2                                                 7.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD001.D14.C2                                                3.0                                                                                                        
LAMI.RD001.D28.C1                                                0.0                                                                   

In [14]:
# Load the QIIME 2 artifact and extract the table data
table_artifact = q2.Artifact.load('../Data/16S/CTF/174950_rarefied_table_unannotated_absolute_abundances_T.qza')
table = table_artifact.view(biom.Table)
    
# Filter the table to remove unwanted samples
biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(table, samples_to_remove)
    
# Check the samples that were removed
dropped_samples = set(original_samples) - set(filtered_samples)
    
print(f"Number of samples in original table: {len(original_samples)}")
print(f"Number of samples after filtering: {len(filtered_samples)}")
print(f"Samples that were removed: {dropped_samples}")

# Convert the filtered table back to a QIIME 2 Artifact
biom_tbl_filtered_artifact = q2.Artifact.import_data('FeatureTable[Frequency]', biom_tbl_filtered)
    
# Save the artifact as a .qza file
biom_tbl_filtered_artifact.save(f'../Data/16S/CTF/filtered_174950_T_10252024.qza')

Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD314.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3'}


'../Data/16S/CTF/filtered_174950_T_10252024.qza'

In [17]:
# Load the distance matrix artifact
table =q2.Artifact.load('../Data/16S/CTF/174951_rarefied_table_unannotated_absolute_abundances_T.qza')
data = table.view(pd.DataFrame)
print(data)

                   TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG  \
LAMI.RD001.D0.C1                                                 0.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD004.D0.C2                                                 0.0                                                                                                        
LAMI.RD001.D0.C2                                                 0.0                                                                                                        
LAMI.RD004.D28.C1                                                1.0                                                                   

In [18]:
# Load the distance matrix artifact
table =q2.Artifact.load('../Data/16S/CTF/filtered_174951_T_10252024.qza')
data = table.view(pd.DataFrame)
print(data)

                   TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG  \
LAMI.RD001.D0.C1                                                 0.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD004.D0.C2                                                 0.0                                                                                                        
LAMI.RD001.D0.C2                                                 0.0                                                                                                        
LAMI.RD004.D28.C1                                                1.0                                                                   

In [23]:
table =q2.Artifact.load('../Data/16S/CTF/174951_rarefied_table_unannotated_absolute_abundances_T.qza')
data = table.view(pd.DataFrame)
print(data)

                   TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG  \
LAMI.RD001.D0.C1                                                 0.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD004.D0.C2                                                 0.0                                                                                                        
LAMI.RD001.D0.C2                                                 0.0                                                                                                        
LAMI.RD004.D28.C1                                                1.0                                                                   

In [21]:
from biom import load_table
from biom.util import biom_open

In [24]:
output_path = '../Data/16S/CTF/174951_rarefied_table_unannotated_absolute_abundances_T.biom'
table = biom.table.Table(data.values, observation_ids=data.index, sample_ids=data.columns)
with biom_open(output_path, 'w') as f:
    table.to_hdf5(f, "transposed table")